In [ ]:
### uniform grid|
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from scipy.ndimage import gaussian_filter
import gaussian_random_fields as gr
from time import time
from timeit import default_timer
from scipy.interpolate import interp2d
import scipy.io as scio
import scipy.sparse as scipa 
import scipy.sparse.linalg as scilg
import cmath as cm
from scipy.interpolate import RegularGridInterpolator
import random

# Set seed
# random.seed(2025)
# Parameter configuration for generating MT forward modeling training/test data
np_dtype = np.float32 # Numerical precision (float32 for efficiency)
n_sample = 50000       # Number of samples (conductivity structures) to generate
n_freq = 16            # Number of frequencies (for cross-frequency generalization validation)
nza = 10               # Number of air layer grid cells (above surface, very low conductivity)
size_b = 10            # Number of boundary region grid cells (for boundary condition handling)
size_k = 64            # Number of core region grid cells (main subsurface simulation region)
alpha_l = [4.0,5.0,6.0,7.0,8.0]  # Gaussian Random Field (GRF) length scale parameters (control smoothness of conductivity structures)
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
def generate_model(alpha_l, n_sample, n_freq, nza, size_b, size_k, device=None, dtype=torch.float32):
    """
    Generate conductivity models and grid parameters for MT forward modeling.

    Builds non-uniform grids with air layer, core region, and boundary regions.
    Generates random conductivity structures via Gaussian Random Fields (GRF) that
    follow geological patterns. Operates entirely in raw conductivity value space,
    using tensor concatenation + torch.where for differentiable extension, aligned
    with extend_model_boundary_single1.

    Args:
        alpha_l: GRF length scales (controls structural smoothness)
        n_sample: Number of model samples to generate
        n_freq: Number of frequencies
        nza: Number of air layer grid cells
        size_b: Number of boundary region grid cells
        size_k: Number of core region grid cells
    Returns:
        zn/yn: Full Z/Y direction grid coordinates; freq: frequency array; ry: observation points;
        model: Full conductivity model; zn0/yn0: core region grids; model_k: core region conductivity
    """
    # Core region depth and horizontal extent (meters)
    z = 75e3  # Core region depth (75 km)
    y = 75e3  # Core region horizontal extent (-75 km to 75 km)

    # Boundary region expansion multipliers (non-uniform grid: sparse boundaries, dense core)
    multiple_t = 1.5  # Top (air layer) expansion multiplier
    multiple_b = 2.0  # Bottom boundary expansion multiplier
    multiple_l = 2.5  # Left boundary expansion multiplier
    multiple_r = 2.5  # Right boundary expansion multiplier

    # Generate frequency array (log-spaced, 10 Hz to 0.01 Hz, MT typical range)
    freq = np.logspace(np.log10(10), np.log10(1/100), n_freq)

    # Build Z-direction grid (vertical: air + core + bottom boundary)
    z_air = -(np.logspace(np.log10(50e3), np.log10(50e3 + multiple_t * z), nza + 1) - 50e3)[::-1]
    zn0 = np.linspace(0, z, size_k + 1)
    z_b = np.logspace(np.log10(zn0[-1]), np.log10(multiple_b * zn0[-1]), size_b + 1)
    zn = np.concatenate((z_air[:-1], zn0, z_b[1:]))

    # Build Y-direction grid (horizontal: left boundary + core + right boundary)
    yn0 = np.linspace(-y, y, size_k + 1)
    y_l = -(np.logspace(np.log10(multiple_l * yn0[-1]), np.log10(yn0[-1]), size_b + 1))
    y_r = np.logspace(np.log10(yn0[-1]), np.log10(multiple_r * yn0[-1]), size_b + 1)
    yn = np.concatenate((y_l[:-1], yn0, y_r[1:]))
    ry = yn0  # Observation point positions

    # Compute full grid dimensions
    len_z = nza + size_b + size_k  # Total Z-direction grid cells
    len_y = 2 * size_b + size_k    # Total Y-direction grid cells

    # ========== Align with extend_model_boundary_single1: define frozen boundary constants ==========
    sig_bound = torch.tensor(1e-2, device=device, dtype=dtype, requires_grad=False)
    sig_air = torch.tensor(1e-9, device=device, dtype=dtype, requires_grad=False)

    # GRF conductivity range (raw values): corresponding to log10 values sig_up=-4 (1e-4), sig_down=0 (1e0)
    sig_up_original = 1e-4  # High resistivity raw value
    sig_down_original = 1e0  # Low resistivity raw value

    # ========== Initialize batch model container ==========
    model = torch.zeros((n_sample, len_z, len_y), device=device, dtype=dtype)

    # Generate conductivity model for each sample
    for ii in range(n_sample):
        model0 = 0.0  # Initialize core region conductivity

        # Superpose GRFs with different length scales for diverse geological structures
        for alpha in alpha_l:
            model_temp0 = gr.gaussian_random_field(
                alpha=alpha, size=size_k, mode='bound', set_1=-4, set_2=0  # Still generates log10 values
            )
            model0 += model_temp0  # Superpose multiple GRFs

        # Smooth and normalize the generated conductivity field to a reasonable range
        min0 = np.min(model_temp0)  # Raw GRF (log10) minimum
        max0 = np.max(model_temp0)  # Raw GRF (log10) maximum
        model0 = gaussian_filter(model0, sigma=2) / len(alpha_l)  # Gaussian smooth and average
        min1 = np.min(model0)  # Smoothed minimum (log10)
        max1 = np.max(model0)  # Smoothed maximum (log10)

        # Remap to original GRF log10 range to preserve conductivity variation amplitude
        model0 = (model0 - min1) * ((max0 - min0) / (max1 - min1 + 1e-12)) + min0  # +1e-12 prevents division by zero

        # ========== Convert log10 values to raw conductivity values ==========
        model0 = torch.tensor(model0, device=device, dtype=dtype, requires_grad=False)
        model0 = torch.pow(10.0, model0)  # log10 -> raw conductivity

        # ========== Step 1: Tensor concatenation to build base model (no in-place ops) ==========
        model0_const_surface = model0.clone()
        model0_const_surface[0, :] = sig_bound  # Set surface row to constant boundary value

        # Y-direction concat: left boundary + core + right boundary
        left_y_pad = sig_bound.expand(size_k, size_b).contiguous()
        right_y_pad = sig_bound.expand(size_k, size_b).contiguous()
        y_padded = torch.cat([left_y_pad, model0, right_y_pad], dim=1).contiguous()

        # Z-direction concat: air + core (after Y concat) + bottom boundary
        air_z_pad = sig_air.expand(nza, len_y).contiguous()
        bottom_z_pad = sig_bound.expand(size_b, len_y).contiguous()
        model_sample = torch.cat([air_z_pad, y_padded, bottom_z_pad], dim=0).contiguous()

        # ========== Step 2: Boundary smoothing interpolation (torch.where + mask, no in-place assignment) ==========
        num_interp = size_b - int(2 * size_b / 3)
        start_idx = int(2 * size_b / 3)
        weight = torch.linspace(0.0, 1.0, num_interp, device=device, dtype=dtype).contiguous()
        weight_flip = weight.flip(0).contiguous()

        # 2.1 Left boundary interpolation
        core_edge_l = model_sample[nza:, size_b].unsqueeze(1).expand(-1, num_interp).contiguous()
        left_interp = core_edge_l * weight + sig_bound * weight_flip

        left_mask = torch.zeros_like(model_sample, dtype=torch.bool, device=device)
        left_mask[nza:, start_idx:size_b] = True
        left_interp_expanded = torch.zeros_like(model_sample, device=device, dtype=dtype)
        left_interp_expanded[nza:, start_idx:size_b] = left_interp
        model_sample = torch.where(left_mask, left_interp_expanded, model_sample)

        # 2.2 Right boundary interpolation
        core_edge_r = model_sample[nza:, -size_b-1].unsqueeze(1).expand(-1, num_interp).contiguous()
        right_interp = core_edge_r * weight_flip + sig_bound * weight

        right_mask = torch.zeros_like(model_sample, dtype=torch.bool, device=device)
        right_mask[nza:, -size_b:-start_idx] = True
        right_interp_expanded = torch.zeros_like(model_sample, device=device, dtype=dtype)
        right_interp_expanded[nza:, -size_b:-start_idx] = right_interp
        model_sample = torch.where(right_mask, right_interp_expanded, model_sample)

        # 2.3 Bottom boundary interpolation
        core_edge_b = model_sample[-size_b-1, :].unsqueeze(0).expand(num_interp, -1).contiguous()
        bottom_interp = core_edge_b * weight_flip.unsqueeze(1) + sig_bound * weight.unsqueeze(1)

        bottom_mask = torch.zeros_like(model_sample, dtype=torch.bool, device=device)
        bottom_mask[-size_b:-start_idx, :] = True
        bottom_interp_expanded = torch.zeros_like(model_sample, device=device, dtype=dtype)
        bottom_interp_expanded[-size_b:-start_idx, :] = bottom_interp
        model_sample = torch.where(bottom_mask, bottom_interp_expanded, model_sample)

        # ========== Step 3: Ensure consistent gradient state (batch model assignment) ==========
        model_sample.requires_grad_(False)
        model[ii] = model_sample.contiguous()

    # ========== Extract core region conductivity model (raw values) ==========
    model_k = model[:, nza:nza+size_k, size_b:size_b+size_k].contiguous()

    return zn, yn, freq, ry, model, zn0, yn0, model_k


zn, yn, freq, ry, sig,zn0,yn0,sig_k,\
         = generate_model(alpha_l,n_sample,n_freq,nza,size_b,size_k,device=device)
# sig_k.shape
# Ensure save directory exists
import os
save_dir = "images"
plt.rcParams['font.family'] = 'Times New Roman'
os.makedirs(save_dir, exist_ok=True)
start_inx=0
end_inx=5
# Loop over sample indices
for idx in range(start_inx, end_inx):
    # Create 1-row 2-column figure
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))
    # ---------------------- Subplot 1: with extended grid (full model) ----------------------
    Y1, Z1 = np.meshgrid(yn, zn)  # Extended coordinate grid
    h1 = ax1.pcolormesh(
        Y1 / 1e3, Z1 / 1e3, (1/sig[idx].detach().cpu().numpy()),  # Conductivity from full model sig
        norm=colors.LogNorm(vmin=1e0, vmax=1e4),  # Unified log normalization
        edgecolors='w', linewidth=0.5,
        cmap='jet', shading='auto'
    )
    ax1.invert_yaxis()  # Invert vertical axis (surface at top, depth at bottom)
    ax1.set_xlabel("Distance (km)", fontsize=20)
    ax1.set_ylabel("Depth (km)", fontsize=20)
    ax1.set_title("(a)",fontsize=24)
    # ---------------------- Subplot 2: without extended grid (core region only) ----------------------
    Y2, Z2 = np.meshgrid(yn0, zn0)  # Core-only coordinate grid
    sig_k[idx][0,...]=sig_k[idx][1,...]
    h2 = ax2.pcolormesh(
        Y2 / 1e3, Z2 / 1e3, (1/sig_k[idx].detach().cpu().numpy()),  # Conductivity from core sig_k
        norm=colors.LogNorm(vmin=1e0, vmax=1e4),  # Same color scale as subplot 1
        cmap='jet', shading='auto'
    )
    ax2.invert_yaxis()
    ax2.set_xlabel("Distance (km)", fontsize=20)
    ax2.set_ylabel("Depth (km)", fontsize=20)
    ax2.set_title("(b)",fontsize=24)

    # ---------------------- Unified colorbar ----------------------
    cbar = fig.colorbar(h1, ax=[ax1, ax2], orientation='horizontal', shrink=0.5, pad=0.12)
    cbar.set_label(
        r'$\log_{10}\,\rho\,(\Omega m)$', 
        fontsize=20
    )
    # ---------------------- Save figure ----------------------
    save_path1 = os.path.join(save_dir, f"model_compare_{idx:03d}.png")
    save_path2 = os.path.join(save_dir, f"model_compare_{idx:03d}.eps")
    plt.savefig(save_path1, dpi=600, bbox_inches='tight')
    plt.savefig(save_path2, dpi=600, bbox_inches='tight')
    plt.close(fig)  # Close figure to free memory


In [12]:
sig_k.shape
sig_k1=sig_k.detach().cpu().numpy()
sig_k1=sig_k1.reshape(n_sample, 1,size_k, size_k)
sig_k1.shape
np.savez(f"sig_model_{n_sample}.npz", sig_model=sig_k1)


In [ ]:
def extend_model_boundary_single1(model0):
    """
    Differentiable model boundary extension (no in-place ops, uses torch.where + tensor concatenation).

    Args:
        model0: Core layer conductivity model (size_k x size_k)
    Returns:
        model: Expanded conductivity model (preserves full gradient)
    """
    if not model0.requires_grad:
        print("Warning: model0.requires_grad=False, output model cannot track gradients!")

    device = model0.device
    dtype = model0.dtype
    nza=10
    size_b=10
    size_k=64

    # Boundary constants (frozen, no gradient)
    sig_bound = torch.tensor(1e-2, device=device, dtype=dtype, requires_grad=False)
    sig_air = torch.tensor(1e-9, device=device, dtype=dtype, requires_grad=False)

    # Compute full grid dimensions
    len_z = nza + size_b + size_k
    len_y = 2 * size_b + size_k

    # ---------------------- Step 1: Build base model (tensor concat, differentiable) ----------------------
    # Y-direction concat: left boundary + core + right boundary
    core_y = model0

    model0_const_surface = core_y.clone()
    model0_const_surface[0, :] = sig_bound
    left_y_pad = sig_bound.expand(size_k, size_b).contiguous()
    right_y_pad = sig_bound.expand(size_k, size_b).contiguous()
    y_padded = torch.cat([left_y_pad, core_y, right_y_pad], dim=1).contiguous()

    # Z-direction concat: air + core (after Y concat) + bottom boundary
    air_z_pad = sig_air.expand(nza, len_y).contiguous()
    bottom_z_pad = sig_bound.expand(size_b, len_y).contiguous()
    model = torch.cat([air_z_pad, y_padded, bottom_z_pad], dim=0).contiguous()

    # ---------------------- Step 2: Boundary smoothing interpolation (torch.where, no in-place ops) ----------------------
    num_interp = size_b - int(2 * size_b / 3)
    start_idx = int(2 * size_b / 3)
    weight = torch.linspace(0.0, 1.0, num_interp, device=device, dtype=dtype).contiguous()
    weight_flip = weight.flip(0).contiguous()

    # 2.1 Left boundary interpolation
    core_edge_l = model[nza:, size_b].unsqueeze(1).expand(-1, num_interp).contiguous()
    left_interp = core_edge_l * weight + sig_bound * weight_flip

    left_mask = torch.zeros_like(model, dtype=torch.bool, device=device)
    left_mask[nza:, start_idx:size_b] = True
    left_interp_expanded = torch.zeros_like(model, device=device, dtype=dtype)
    left_interp_expanded[nza:, start_idx:size_b] = left_interp
    model = torch.where(left_mask, left_interp_expanded, model)

    # 2.2 Right boundary interpolation
    core_edge_r = model[nza:, -size_b-1].unsqueeze(1).expand(-1, num_interp).contiguous()
    right_interp = core_edge_r * weight_flip + sig_bound * weight

    right_mask = torch.zeros_like(model, dtype=torch.bool, device=device)
    right_mask[nza:, -size_b:-start_idx] = True
    right_interp_expanded = torch.zeros_like(model, device=device, dtype=dtype)
    right_interp_expanded[nza:, -size_b:-start_idx] = right_interp
    model = torch.where(right_mask, right_interp_expanded, model)

    # 2.3 Bottom boundary interpolation
    core_edge_b = model[-size_b-1, :].unsqueeze(0).expand(num_interp, -1).contiguous()
    bottom_interp = core_edge_b * weight_flip.unsqueeze(1) + sig_bound * weight.unsqueeze(1)

    bottom_mask = torch.zeros_like(model, dtype=torch.bool, device=device)
    bottom_mask[-size_b:-start_idx, :] = True
    bottom_interp_expanded = torch.zeros_like(model, device=device, dtype=dtype)
    bottom_interp_expanded[-size_b:-start_idx, :] = bottom_interp
    model = torch.where(bottom_mask, bottom_interp_expanded, model)

    # ---------------------- Step 3: Ensure consistent gradient tracking state ----------------------
    model.requires_grad_(model0.requires_grad)

    return model.contiguous()

In [ ]:
dtype = torch.float32

# 2. Call generate_model to generate models
zn, yn, freq, ry, sig, zn0, yn0, sig_k = generate_model(
    alpha_l=alpha_l,
    n_sample=n_sample,
    n_freq=n_freq,
    nza=nza,
    size_b=size_b,
    size_k=size_k,
    device=device,
    dtype=dtype
)

# 3. Call modified extend_model_boundary_single1 with unified parameters
sig1 = extend_model_boundary_single1(sig_k[0,...])

# 4. Recompute error
cha1 = sig1 - sig[0,...]
print("Error min:", cha1.min())
print("Error max:", cha1.max())
print("Error abs mean:", torch.abs(cha1).mean())

In [78]:
print(zn.shape)
print(yn.shape)
print(freq.shape)
print(sig.shape)
print(zn0.shape)
print(yn0.shape)
print(sig_k.shape)
# sig_k=sig_k.detach().cpu().numpy()
# sig_model=sig_k.reshape(n_sample, 1,size_k, size_k)
# sig_model.shape
# np.savez(f'sig_model_{size_k}_{n_sample}.npz',sig_model= sig_model)
n_freq

(85,)
(85,)
(16,)
torch.Size([5, 84, 84])
(65,)
(65,)
torch.Size([5, 64, 64])


16

In [79]:
from MT2D_secondary_direct1 import *
from MT2D_secondary_direct import *
n_freq = np.size(freq)
n_ry   = len(ry)
channel=4
obs_data= np.zeros((n_sample,channel,n_freq,n_ry),dtype=np_dtype)
obs_data1=torch.zeros((n_sample,channel,n_freq,n_ry),dtype=torch.float32,device=device)

time0 = default_timer()
result = []
for ii in range(n_sample):
    model = MT2DFD(nza, zn, yn, freq, ry, sig[ii, :, :].detach().cpu().numpy())
    model1=MT2DFD1(nza, zn, yn, freq, ry, sig[ii, :, :])
    obs_data[ii, 0, :, :], obs_data[ii, 1, :, :], obs_data[ii, 2, :, :], obs_data[ii, 3, :, :] = model.mt2d(mode="TETM")
    obs_data1[ii, 0, :, :], obs_data1[ii, 1, :, :], obs_data1[ii, 2, :, :], obs_data1[ii, 3, :, :] = model1.mt2d(mode="TETM")
    print(f"{ii} of {n_sample} finished!")
time1 = default_timer()
print(f"Time used for forward modeling: {time1 - time0} seconds")


0 of 5 finished!
1 of 5 finished!
2 of 5 finished!
3 of 5 finished!
4 of 5 finished!
Time used for forward modeling: 56.870754500007024 seconds


In [ ]:
import matplotlib.pyplot as plt
fig, axs = plt.subplots(1, 2, figsize=(10, 5))
im1 = axs[1].imshow(np.log10(abs(obs_data[0, 2, :, :])), cmap='jet')
im0 = axs[0].imshow(np.log10(abs(obs_data1[0, 2, :, :]).detach().cpu().numpy()), cmap='jet')

In [81]:
cha=obs_data[0,:, :, :]-obs_data1[0, :, :, :].detach().cpu().numpy()
cha.min(),cha.max()
obs_data1.shape

torch.Size([5, 4, 16, 65])

In [128]:
sig.shape
obs_data.shape

(5, 4, 16, 65)

In [82]:
sig_model=sig_k.reshape(n_sample, 1,size_k, size_k)
sig_model=sig_model.detach().cpu().numpy()
print(sig_model.shape)
obs_data=obs_data1.detach().cpu().numpy()# np.save(f'sig_model_{n_sample}.npy',sig_model= sig_model)
np.savez(f'obs_modelanddata_{64}_{n_sample}.npz',sig_model=sig_model, obs_data= obs_data)


(5, 1, 64, 64)


In [37]:
sig_k[0,:, :]

tensor([[0.1093, 0.1055, 0.0978,  ..., 0.1093, 0.1099, 0.1102],
        [0.1093, 0.1055, 0.0978,  ..., 0.1093, 0.1099, 0.1102],
        [0.1236, 0.1192, 0.1105,  ..., 0.1289, 0.1292, 0.1293],
        ...,
        [0.0230, 0.0227, 0.0219,  ..., 0.0257, 0.0259, 0.0259],
        [0.0294, 0.0289, 0.0277,  ..., 0.0322, 0.0325, 0.0326],
        [0.0338, 0.0332, 0.0317,  ..., 0.0366, 0.0370, 0.0372]])

In [86]:
import numpy as np
sig_model1=np.load(f'waiyan_sig.npz')["waiyan1"]
sig_k1=np.load(f'waiyan_sig.npz')["sig_model"]
# sig_k=np.load(f'obs_modelanddata_64_5.npz')["sig_model"]

# mt_obs=np.load(f'waiyan_sig.npz')["mt_obs"]
sig_k1.shape,sig_model1.shape


((64, 64), (84, 84))

In [89]:
sig_k.shape

torch.Size([5, 64, 64])

In [ ]:
# cha2=sig_k1-sig_k[0,...].detach().cpu().numpy()
# print(cha2.min()),print(cha2.max())
# cha3=sig_model1-sig[0,...].detach().cpu().numpy()
# print(cha3.min()),print(cha3.max())


-5.9604645e-08
5.9604645e-08
-5.9604645e-08
5.9604645e-08


(None, None)

In [96]:
sig[0,...].shape

torch.Size([84, 84])

In [138]:
sig3=sig[0,:,:].detach().cpu().numpy()
cha3=sig_model1-sig3
cha3.min()

np.float32(0.0)